# Relation Networks for Few-Shot Image Classification with Explainable AI (XAI)

## 📋 Research Paper-Level Documentation

### Abstract
This notebook implements a Relation Network-based few-shot learning system integrated with Explainable AI (XAI) techniques. Unlike Siamese networks that use fixed distance metrics, Relation Networks learn the comparison function using a neural network, allowing them to adaptively learn which features are most important for comparing images.

### 1. Introduction
Relation Networks (RN) are a class of neural networks designed for few-shot learning that learn to compare pairs of images using a learned similarity function. The key innovation is that the comparison operation itself is learned rather than using a fixed metric.

**Key Difference from Siamese Networks:**
- **Siamese Networks**: Use fixed distance metrics (e.g., Euclidean, cosine similarity)
- **Relation Networks**: Learn the comparison function using a neural network (relation module)

**Problem Setup:**
- **N-way K-shot classification**: Given N classes with K support examples each
- **Support set**: Examples used for computing relation scores
- **Query set**: Examples to be classified based on learned relation scores

### 2. Mathematical Formulation

#### 2.1 Feature Encoder
The feature encoder $f_\phi$ maps images to feature maps:

$$\mathbf{F} = f_\phi(\mathbf{x}) \in \mathbb{R}^{C \times H \times W}$$

where $C$ is the number of channels, and $H, W$ are spatial dimensions.

#### 2.2 Feature Concatenation
For comparing query $\mathbf{x}_q$ and support $\mathbf{x}_s$:

$$\mathbf{C} = [f_\phi(\mathbf{x}_q); f_\phi(\mathbf{x}_s)] \in \mathbb{R}^{2C \times H \times W}$$

where $[\cdot; \cdot]$ denotes channel-wise concatenation.

#### 2.3 Relation Module
The relation module $g_\theta$ learns to output a relation score:

$$r = g_\theta(\mathbf{C}) \in [0, 1]$$

The relation score $r$ indicates how similar the query and support images are.

#### 2.4 Classification via Relation Aggregation
For N-way classification, aggregate relation scores by class:

$$\text{score}(c) = \sum_{i: y_i = c} r_i$$

$$P(y=c | \mathbf{x}_q) = \frac{\exp(\text{score}(c))}{\sum_{c'} \exp(\text{score}(c'))}$$

#### 2.5 Loss Function
Mean Squared Error (MSE) for relation learning:

$$L_{relation} = \frac{1}{N} \sum (r_{pred} - r_{target})^2$$

where $r_{target} = 1$ if same class, $0$ otherwise.

![Few-Shot Learning Architecture](few%20shot%20image%20classification.png)

**Figure: Relation Network Architecture** - The model encodes both query and support images, concatenates their features, and passes them through a learned relation module to produce relation scores. These scores are aggregated by class for final classification.

In [ ]:
# ============================================================
# IMPORTS AND CONFIGURATION
# ============================================================

import os
import random
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, 
    precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedShuffleSplit
from scipy.stats import ttest_ind, sem
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for publication-quality plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"{'='*60}")
print(f"Device Configuration")
print(f"{'='*60}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cudnn.benchmark = True
print(f"Using: {DEVICE}")
print(f"{'='*60}\n")

## 3. Configuration and Hyperparameters

### 3.1 Experimental Setup

| Parameter | Value | Description |
|-----------|-------|-------------|
| N-way | 8 | Number of classes per episode |
| K-shot | 5 | Support examples per class |
| Q-query | 15 | Query examples per class |
| Feature dim | 64 | Feature map channels |
| Hidden dim | 128 | Relation module hidden dimension |
| Learning rate | 1e-3 | Initial learning rate |
| Weight decay | 1e-4 | L2 regularization |
| Epochs | 30 | Training iterations |

In [ ]:
# ============================================================
# HYPERPARAMETERS AND PATHS
# ============================================================

# Data paths - MODIFY THESE FOR YOUR DATASET
DATA_ROOT = '/kaggle/input/cucumber-dataset/Original Image'
OUTPUT_DIR = '/kaggle/working'

# Create output directories
SPLIT_DIR = os.path.join(OUTPUT_DIR, 'splits')
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')
XAI_DIR = os.path.join(OUTPUT_DIR, 'xai')
CKPT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')

for p in [SPLIT_DIR, PLOTS_DIR, XAI_DIR, CKPT_DIR]: 
    os.makedirs(p, exist_ok=True)

# Random seed for reproducibility
RNG_SEED = 42
torch.manual_seed(RNG_SEED)
np.random.seed(RNG_SEED)
random.seed(RNG_SEED)

# Dataset splitting ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Few-shot learning parameters
N_WAY = 8           # Number of classes per episode
K_SHOT = 5          # Support examples per class
Q_QUERY = 15        # Query examples per class

# Training parameters
EPISODES_PER_EPOCH = 20
VAL_EPISODES = 10
TEST_EPISODES = 20
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# Model parameters
FEATURE_DIM = 64       # Feature encoder output channels
HIDDEN_DIM = 128       # Relation module hidden dimension
IMAGE_SIZE = 128

print(f"{'='*60}")
print(f"Experimental Configuration")
print(f"{'='*60}")
print(f"N-way K-shot: {N_WAY}-way {K_SHOT}-shot")
print(f"Episodes per epoch: {EPISODES_PER_EPOCH}")
print(f"Total epochs: {NUM_EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"{'='*60}\n")

## 4. Data Handling

### 4.1 Stratified Data Splitting

In [ ]:
# ============================================================
# DATA SPLITTING FUNCTIONS
# ============================================================

def make_stratified_splits(root_dir, seed=RNG_SEED):
    """
    Perform stratified train/val/test splitting.
    
    Mathematical formulation:
    For each class c with n_c samples:
        n_train_c = floor(n_c * 0.8)
        n_val_c = floor(n_c * 0.1)
        n_test_c = n_c - n_train_c - n_val_c
    """
    data = []
    root = Path(root_dir)
    
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    print(f"Discovered {len(classes)} classes: {classes}")
    
    if len(classes) < 2:
        raise ValueError('Dataset root must contain at least 2 class subfolders')
    
    for lbl, cls in enumerate(classes):
        images = list((root / cls).glob('*'))
        images = [x for x in images if x.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tif']]
        for img in images:
            data.append({'image': str(img), 'label': lbl, 'class': cls})
    
    df = pd.DataFrame(data)
    print(f"Total images: {len(df)}")
    print(f"Images per class: {df['label'].value_counts().sort_index().tolist()}")
    
    x, y = df['image'], df['label']
    
    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=(1-TRAIN_RATIO), random_state=seed)
    train_idx, temp_idx = next(splitter1.split(x, y))
    
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_temp = df.iloc[temp_idx].reset_index(drop=True)
    
    test_ratio_adjusted = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=test_ratio_adjusted, random_state=seed)
    val_idx, test_idx = next(splitter2.split(df_temp['image'], df_temp['label']))
    
    df_val = df_temp.iloc[val_idx].reset_index(drop=True)
    df_test = df_temp.iloc[test_idx].reset_index(drop=True)
    
    df_train.to_csv(os.path.join(SPLIT_DIR, 'train.csv'), index=False)
    df_val.to_csv(os.path.join(SPLIT_DIR, 'val.csv'), index=False)
    df_test.to_csv(os.path.join(SPLIT_DIR, 'test.csv'), index=False)
    
    print(f"\nStratified Split Results:")
    print(f"  Training:   {len(df_train)} images ({len(df_train)/len(df)*100:.1f}%)")
    print(f"  Validation: {len(df_val)} images ({len(df_val)/len(df)*100:.1f}%)")
    print(f"  Testing:    {len(df_test)} images ({len(df_test)/len(df)*100:.1f}%)")
    
    return df_train, df_val, df_test, classes

# Perform data splitting
df_train, df_val, df_test, CLASS_NAMES = make_stratified_splits(DATA_ROOT)
NUM_CLASSES = len(CLASS_NAMES)
print(f"\nClass names: {CLASS_NAMES}")

In [ ]:
# ============================================================
# DATASET AND TRANSFORMS
# ============================================================

class ImagePathsDataset(Dataset):
    """Custom dataset for loading images from CSV file paths."""
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image']).convert('RGB')
        image = transforms.ToTensor()(image)
        
        if self.transform is not None:
            image = self.transform(image)
        
        return image, int(row['label'])


def make_transforms(img_size=IMAGE_SIZE):
    """Create training and evaluation transforms."""
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, eval_transform

train_transform, eval_transform = make_transforms()

In [ ]:
# ============================================================
# EPISODIC SAMPLER
# ============================================================

class RelationSampler:
    """
    Sampler for creating few-shot episodes for Relation Networks.
    
    For Relation Networks, we compute relation scores between each query
    and all support images to determine classification.
    """
    def __init__(self, labels, n_way, k_shot, q_query, episodes, seed=RNG_SEED):
        self.labels = np.array(labels)
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.episodes = episodes
        self.rng = np.random.RandomState(seed)
        
        self.by_class = {c: np.where(self.labels == c)[0] for c in np.unique(self.labels)}
        
        min_required = k_shot + q_query
        for c, idx in self.by_class.items():
            if len(idx) < min_required:
                raise ValueError(f'Class {c} has {len(idx)} samples, needs at least {min_required}')

    def __len__(self):
        return self.episodes

    def __iter__(self):
        for _ in range(self.episodes):
            selected_classes = self.rng.choice(list(self.by_class.keys()), size=self.n_way, replace=False)
            support_idx = []
            query_idx = []
            
            for c in selected_classes:
                choices = self.rng.choice(self.by_class[c], size=self.k_shot + self.q_query, replace=False)
                support_idx.extend(choices[:self.k_shot].tolist())
                query_idx.extend(choices[self.k_shot:].tolist())
            
            yield support_idx, query_idx


def episode_loader(df, transform, n_way=N_WAY, k_shot=K_SHOT, q_query=Q_QUERY, episodes=EPISODES_PER_EPOCH):
    dataset = ImagePathsDataset(df, transform=transform)
    sampler = RelationSampler(df['label'].to_numpy(), n_way=n_way, k_shot=k_shot, q_query=q_query, episodes=episodes)
    return dataset, sampler

## 5. Relation Network Architecture

### 5.1 Feature Encoder

**Architecture Design:**

| Layer | Output Channels | Output Size |
|-------|----------------|-------------|
| Conv1 + BN + ReLU + Pool | 64 | 64×64 |
| Conv2 + BN + ReLU + Pool | 64 | 32×32 |
| Conv3 + BN + ReLU + Pool | 64 | 16×16 |
| Conv4 + BN + ReLU + Pool | 64 | 8×8 |
| Adaptive Avg Pool | 64 | 4×4 |

### 5.2 Relation Module

The relation module takes concatenated features [F_q; F_s] and learns the similarity:

| Layer | Output | Description |
|-------|--------|-------------|
| Conv + BN + ReLU + Pool | 64 | Process concatenated features |
| Conv + BN + ReLU + Pool | 64 | Further processing |
| Conv + BN + ReLU + Pool | 64 | Final conv layer |
| Flatten + FC + ReLU | 128 | MLP hidden layer |
| Dropout | - | Regularization |
| FC + Sigmoid | 1 | Relation score [0,1] |

In [ ]:
# ============================================================
# RELATION NETWORK MODEL
# ============================================================

class FeatureEncoder(nn.Module):
    """
    Convolutional encoder for extracting features from images.
    
    Architecture: 4 convolutional blocks producing spatial feature maps.
    
    Output: [batch_size, 64, 4, 4] feature maps
    """
    def __init__(self):
        super().__init__()

        self.conv_blocks = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((4, 4))
        )

    def forward(self, x):
        return self.conv_blocks(x)


class RelationModule(nn.Module):
    """
    Relation module that learns to compare pairs of images.
    
    Takes concatenated features [F_q; F_s] and outputs relation score.
    
    Mathematical: r = g_theta([F_q; F_s]) where g_theta is learned
    """
    def __init__(self, input_dim=128):
        super().__init__()

        self.relation_network = nn.Sequential(
            nn.Conv2d(input_dim, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 4 * 4, HIDDEN_DIM),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(HIDDEN_DIM, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.relation_network(x)


class RelationNetwork(nn.Module):
    """
    Relation Network for few-shot classification.
    
    Forward pass:
    1. Encode query and support: F_q = f_phi(x_q), F_s = f_phi(x_s)
    2. Concatenate: C = [F_q; F_s]
    3. Compute relation: r = g_theta(C)
    4. Aggregate by class
    5. Classify
    """
    def __init__(self):
        super().__init__()
        self.feature_encoder = FeatureEncoder()
        self.relation_module = RelationModule(input_dim=128)

    def compute_relation_scores(self, query_features, support_features):
        """Compute relation scores between query and support images."""
        Q = query_features.size(0)
        S = support_features.size(0)
        
        # Expand and concatenate
        query_expanded = query_features.unsqueeze(1).expand(Q, S, -1, -1, -1)
        support_expanded = support_features.unsqueeze(0).expand(Q, S, -1, -1, -1)
        concatenated = torch.cat([query_expanded, support_expanded], dim=2)
        
        # Reshape for relation module
        Q2, S2, C2, H, W = concatenated.shape
        concatenated = concatenated.reshape(Q2 * S2, C2, H, W)
        
        # Compute relation scores
        relation_scores = self.relation_module(concatenated)
        relation_scores = relation_scores.view(Q, S)
        
        return relation_scores

    def forward(self, support, support_labels, query):
        """Forward pass for few-shot classification."""
        # Encode all images
        support_features = self.feature_encoder(support)
        query_features = self.feature_encoder(query)
        
        # Compute relation scores
        relation_scores = self.compute_relation_scores(query_features, support_features)
        
        # Aggregate scores by class
        Q = query_features.size(0)
        unique_labels = torch.unique(support_labels)
        N = len(unique_labels)
        
        # Create label mapping
        label_map = {int(c): i for i, c in enumerate(unique_labels)}
        
        # Aggregate scores for each class
        logits = torch.zeros(Q, N, device=query.device)
        
        for idx, c in enumerate(unique_labels):
            mask = (support_labels == c).unsqueeze(0)
            class_scores = (relation_scores * mask.float()).sum(dim=1)
            class_scores = class_scores / (mask.sum().float() + 1e-8)
            logits[:, idx] = class_scores
        
        return logits, query_features, support_features, relation_scores


class RelationLoss(nn.Module):
    """
    MSE loss for relation network training.
    
    L = (1/N) * sum((r_pred - r_target)^2)
    where r_target = 1 if same class, 0 otherwise
    """
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()
    
    def forward(self, relation_scores, query_labels, support_labels):
        Q, S = relation_scores.shape
        query_labels_expanded = query_labels.unsqueeze(1)
        support_labels_expanded = support_labels.unsqueeze(0)
        targets = (query_labels_expanded == support_labels_expanded).float()
        return self.mse(relation_scores, targets)

# Initialize model
model = RelationNetwork().to(DEVICE)
print(f"{'='*60}")
print(f"Model Architecture")
print(f"{'='*60}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Feature encoder output: [batch, 64, 4, 4]")
print(f"Relation module output: [batch, 1] (relation score in [0,1])")
print(f"{'='*60}\n")

## 6. Evaluation Metrics

### 6.1 Mathematical Formulations

**Accuracy:**
$$\text{Accuracy} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}$$

**F1-Score (Macro):**
$$F1_{\text{macro}} = \frac{1}{C} \sum_{c=1}^{C} F1_c$$

**ECE (Expected Calibration Error):**
$$ECE = \sum_{b=1}^{B} \frac{|B_b|}{n} |\text{acc}(B_b) - \text{conf}(B_b)|$$

**Attribution Sparsity:**
$$Sparsity = \frac{1}{H \cdot W} \sum_{i,j} \mathbb{1}(|a_{ij}| < \tau)$$

where $\tau = 0.6$.

In [ ]:
# ============================================================
# METRICS COMPUTATION
# ============================================================

def compute_ece(probs, labels, n_bins=15):
    """Compute Expected Calibration Error (ECE)."""
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(labels)
    
    ece = torch.zeros(1, device=probs.device)
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    
    for i in range(n_bins):
        in_bin = confidences.gt(bin_boundaries[i]) & confidences.le(bin_boundaries[i + 1])
        prop_in_bin = in_bin.float().mean()
        
        if prop_in_bin.item() > 0:
            accuracy_in_bin = accuracies[in_bin].float().mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += torch.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    
    return ece.item()


def compute_attribution_sparsity(attributions, threshold=0.6):
    """Compute attribution sparsity of XAI explanation maps."""
    attr = np.abs(attributions)
    if attr.max() > 0:
        attr = attr / attr.max()
    return np.mean(attr < threshold)


def compute_all_metrics(y_true, y_pred, y_prob, class_names=None):
    """Compute comprehensive evaluation metrics."""
    y_prob_np = y_prob.cpu().numpy() if isinstance(y_prob, torch.Tensor) else y_prob
    
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    precision, recall, f1_per_class, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    
    ece_val = compute_ece(torch.from_numpy(y_prob_np), torch.from_numpy(y_true), n_bins=15)
    cm = confusion_matrix(y_true, y_pred)
    
    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_micro': f1_micro,
        'f1_weighted': f1_weighted,
        'precision_per_class': precision.tolist(),
        'recall_per_class': recall.tolist(),
        'f1_per_class': f1_per_class.tolist(),
        'ece': ece_val,
        'confusion_matrix': cm.tolist(),
    }


def print_metrics(metrics, class_names=None):
    """Pretty print metrics."""
    print(f"\n{'='*60}")
    print("EVALUATION METRICS")
    print(f"{'='*60}")
    print(f"Overall Accuracy:  {metrics['accuracy']:.4f}")
    print(f"F1-Score (Macro):  {metrics['f1_macro']:.4f}")
    print(f"F1-Score (Micro):  {metrics['f1_micro']:.4f}")
    print(f"F1-Score (Weighted): {metrics['f1_weighted']:.4f}")
    print(f"ECE (Calibration): {metrics['ece']:.4f}")
    print(f"\nPer-Class F1 Scores:")
    
    if class_names:
        for i, (name, f1) in enumerate(zip(class_names, metrics['f1_per_class'])):
            print(f"  Class {i} ({name[:15]:15s}): {f1:.4f}")
    else:
        for i, f1 in enumerate(metrics['f1_per_class']):
            print(f"  Class {i}: {f1:.4f}")
    print(f"{'='*60}\n")

## 7. XAI Methods

### 7.1 Grad-CAM and Saliency Maps

We integrate two XAI techniques for interpreting Relation Network predictions:

**1. Grad-CAM (Gradient-weighted Class Activation Mapping)**

$$L_{Grad-CAM}^{(c)} = ReLU\left(\sum_{k} \alpha_k^c \cdot A^k\right)$$

where $\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A_{ij}^k}$

**2. Saliency Maps**

$$S_{ij} = \left| \frac{\partial y^c}{\partial x_{ij}} \right|$$

In [ ]:
# ============================================================
# XAI METHODS
# ============================================================

class RelationGradCAM:
    """Grad-CAM for Relation Networks."""
    def __init__(self, model, target_layer, support_images=None, support_labels=None):
        self.model = model
        self.target_layer = target_layer
        self.support_images = support_images
        self.support_labels = support_labels
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()
    
    def _register_hooks(self):
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        
        def forward_hook(module, inp, out):
            self.activations = out.detach()
        
        self.hook_handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.hook_handles.append(self.target_layer.register_full_backward_hook(backward_hook))
    
    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        self.model.zero_grad()
        
        if self.support_images is None or self.support_labels is None:
            raise ValueError("GradCAM requires support_images and support_labels")
        
        query_img = input_tensor.unsqueeze(0)
        logits, query_emb, support_emb, similarities = self.model(
            self.support_images, self.support_labels, query_img
        )
        
        if target_class is None:
            target_class = torch.argmax(logits, dim=1).item()
        
        if target_class >= logits.shape[1]:
            target_class = torch.argmax(logits, dim=1).item()
        
        score = logits[0, target_class]
        score.backward(retain_graph=True)
        
        grads = self.gradients[0]
        acts = self.activations[0]
        weights = torch.mean(grads, dim=(1, 2), keepdim=True)
        cam = torch.sum(weights * acts, dim=0).cpu().numpy()
        cam = np.maximum(cam, 0)
        cam = cam - np.min(cam)
        if np.max(cam) > 0:
            cam = cam / np.max(cam)
        
        return cam
    
    def close(self):
        for handle in self.hook_handles:
            handle.remove()


def relation_saliency_map(model, input_tensor, support_images=None, support_labels=None, target_class=None):
    """Compute gradient-based saliency map for Relation Network."""
    model.eval()
    input_tensor = input_tensor.unsqueeze(0).clone().detach().requires_grad_(True)
    
    if support_images is None or support_labels is None:
        raise ValueError("saliency_map requires support_images and support_labels")
    
    logits, query_emb, support_emb, similarities = model(
        support_images, support_labels, input_tensor
    )
    
    if target_class is None:
        target_class = torch.argmax(logits, dim=1).item()
    
    if target_class >= logits.shape[1]:
        target_class = torch.argmax(logits, dim=1).item()
    
    score = logits[0, target_class]
    score.backward()
    
    saliency = input_tensor.grad.data.abs().squeeze().cpu().numpy()
    saliency = np.max(saliency, axis=0)
    saliency = saliency - saliency.min()
    if saliency.max() > 0:
        saliency = saliency / saliency.max()
    
    return saliency


def save_heatmap(img, mask, path, alpha=0.5, title=None):
    """Save heatmap overlay visualization."""
    img_np = img.cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_p = np.clip((img_np * std + mean), 0, 1)
    
    cmap = plt.get_cmap('jet')
    heatmap = cmap(mask)[..., :3]
    overlay = np.clip((1 - alpha) * img_p + alpha * heatmap, 0, 1)
    
    plt.figure(figsize=(5, 5))
    plt.axis('off')
    if title:
        plt.title(title)
    plt.imshow(overlay)
    plt.tight_layout(pad=0)
    plt.savefig(path, dpi=150, bbox_inches='tight', pad_inches=0.1, facecolor='white')
    plt.close()

## 8. Training Pipeline

### 8.1 Episode-Based Training

**Training Algorithm:**

```
FOR each epoch:
    FOR each episode:
        1. Sample N classes from training set
        2. Sample K support and Q query images per class
        3. Encode images: F = f_phi(images)
        4. Concatenate query-support pairs: C = [F_q; F_s]
        5. Compute relations: r = g_theta(C)
        6. Compute losses: L = MSE(r, target) + CE(scores, labels)
        7. Update parameters: theta = theta - lr * grad(L)
    END
END
```

In [ ]:
# ============================================================
# TRAINING FUNCTIONS
# ============================================================

def run_relation_episode(model, optimizer, dataset, support_idx, query_idx, relation_criterion):
    """Run a single few-shot episode for Relation Network training."""
    model.train()
    
    support_images = torch.stack([dataset[i][0] for i in support_idx]).to(DEVICE)
    support_labels = torch.tensor([dataset[i][1] for i in support_idx], dtype=torch.long).to(DEVICE)
    query_images = torch.stack([dataset[i][0] for i in query_idx]).to(DEVICE)
    query_labels = torch.tensor([dataset[i][1] for i in query_idx], dtype=torch.long).to(DEVICE)
    
    unique = torch.unique(support_labels)
    label_map = {int(c): i for i, c in enumerate(unique)}
    support_labels_mapped = torch.tensor([label_map[int(l)] for l in support_labels], dtype=torch.long).to(DEVICE)
    query_labels_mapped = torch.tensor([label_map[int(l)] for l in query_labels], dtype=torch.long).to(DEVICE)
    
    logits, query_emb, support_emb, relation_scores = model(
        support_images, support_labels_mapped, query_images
    )
    
    # Relation loss + Classification loss
    rel_loss = relation_criterion(relation_scores, query_labels_mapped, support_labels_mapped)
    ce_loss = F.cross_entropy(logits, query_labels_mapped)
    total_loss = rel_loss + ce_loss
    
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    
    preds = torch.argmax(logits, dim=1)
    acc = (preds == query_labels_mapped).float().mean().item()
    probs = F.softmax(logits, dim=1).detach().cpu().numpy()
    
    return total_loss.item(), acc, preds.detach().cpu().numpy(), query_labels_mapped.detach().cpu().numpy(), probs, rel_loss.item()


def validate_relation_episode(model, dataset, support_idx, query_idx, relation_criterion):
    """Run a single few-shot episode for validation."""
    model.eval()
    
    with torch.no_grad():
        support_images = torch.stack([dataset[i][0] for i in support_idx]).to(DEVICE)
        support_labels = torch.tensor([dataset[i][1] for i in support_idx], dtype=torch.long).to(DEVICE)
        query_images = torch.stack([dataset[i][0] for i in query_idx]).to(DEVICE)
        query_labels = torch.tensor([dataset[i][1] for i in query_idx], dtype=torch.long).to(DEVICE)
        
        unique = torch.unique(support_labels)
        label_map = {int(c): i for i, c in enumerate(unique)}
        support_labels_mapped = torch.tensor([label_map[int(l)] for l in support_labels], dtype=torch.long).to(DEVICE)
        query_labels_mapped = torch.tensor([label_map[int(l)] for l in query_labels], dtype=torch.long).to(DEVICE)
        
        logits, query_emb, support_emb, relation_scores = model(
            support_images, support_labels_mapped, query_images
        )
        
        rel_loss = relation_criterion(relation_scores, query_labels_mapped, support_labels_mapped)
        ce_loss = F.cross_entropy(logits, query_labels_mapped)
        total_loss = rel_loss + ce_loss
        
        preds = torch.argmax(logits, dim=1)
        acc = (preds == query_labels_mapped).float().mean().item()
        probs = F.softmax(logits, dim=1).cpu().numpy()
        
        return total_loss.item(), acc, preds.cpu().numpy(), query_labels_mapped.cpu().numpy(), probs, rel_loss.item()


def train_relation_network(model, df_train, df_val, n_epochs=NUM_EPOCHS, lr=LEARNING_RATE):
    """Complete training loop for Relation Network."""
    train_dataset, train_sampler = episode_loader(df_train, train_transform, episodes=EPISODES_PER_EPOCH)
    val_dataset, val_sampler = episode_loader(df_val, eval_transform, episodes=VAL_EPISODES, k_shot=K_SHOT, q_query=11)
    
    relation_criterion = RelationLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    
    history = {'train_loss': [], 'train_acc': [], 'train_relation_loss': [],
               'val_loss': [], 'val_acc': [], 'val_relation_loss': [], 'epoch_times': []}
    
    best_val_acc = 0.0
    best_ckpt = None
    
    print(f"\n{'='*60}")
    print("TRAINING RELATION NETWORK")
    print(f"{'='*60}\n")
    
    epoch_start = time.time()
    
    for epoch in range(1, n_epochs + 1):
        model.train()
        train_losses, train_accs, train_rel_losses = [], [], []
        
        for support_idx, query_idx in train_sampler:
            loss, acc, _, _, _, rel_loss = run_relation_episode(
                model, optimizer, train_dataset, support_idx, query_idx, relation_criterion
            )
            train_losses.append(loss)
            train_accs.append(acc)
            train_rel_losses.append(rel_loss)
        
        scheduler.step()
        
        model.eval()
        val_losses, val_accs, val_rel_losses = [], [], []
        
        with torch.no_grad():
            for support_idx, query_idx in val_sampler:
                loss, acc, _, _, _, rel_loss = validate_relation_episode(
                    model, val_dataset, support_idx, query_idx, relation_criterion
                )
                val_losses.append(loss)
                val_accs.append(acc)
                val_rel_losses.append(rel_loss)
        
        train_loss = float(np.mean(train_losses))
        train_acc = float(np.mean(train_accs))
        train_rel_loss = float(np.mean(train_rel_losses))
        val_loss = float(np.mean(val_losses))
        val_acc = float(np.mean(val_accs))
        val_rel_loss = float(np.mean(val_rel_losses))
        epoch_time = time.time() - epoch_start
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_relation_loss'].append(train_rel_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_relation_loss'].append(val_rel_loss)
        history['epoch_times'].append(epoch_time)
        
        if epoch % 2 == 0:
            print(f"Epoch {epoch:3d}/{n_epochs} | Train Loss: {train_loss:.4f}, Acc: {train_acc:.3f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.3f} | Time: {epoch_time:.1f}s")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_ckpt = os.path.join(CKPT_DIR, 'best_relation.pth')
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 
                        'optimizer_state_dict': optimizer.state_dict(), 'val_acc': val_acc}, best_ckpt)
        
        epoch_start = time.time()
    
    print(f"\nTraining Complete! Best Val Acc: {best_val_acc:.4f}")
    return history, best_ckpt

In [ ]:
# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_relation_network(model, df_test, episodes=TEST_EPISODES):
    """Evaluate Relation Network on test set."""
    test_dataset, test_sampler = episode_loader(
        df_test, eval_transform, episodes=episodes, k_shot=K_SHOT, q_query=11
    )
    
    relation_criterion = RelationLoss()
    
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    all_loss, all_acc = [], []
    
    with torch.no_grad():
        for support_idx, query_idx in test_sampler:
            support_images = torch.stack([test_dataset[i][0] for i in support_idx]).to(DEVICE)
            support_labels = torch.tensor([test_dataset[i][1] for i in support_idx], dtype=torch.long).to(DEVICE)
            query_images = torch.stack([test_dataset[i][0] for i in query_idx]).to(DEVICE)
            query_labels = torch.tensor([test_dataset[i][1] for i in query_idx], dtype=torch.long).to(DEVICE)
            
            unique = torch.unique(support_labels)
            label_map = {int(c): i for i, c in enumerate(unique)}
            support_labels_mapped = torch.tensor([label_map[int(l)] for l in support_labels], dtype=torch.long).to(DEVICE)
            query_labels_mapped = torch.tensor([label_map[int(l)] for l in query_labels], dtype=torch.long).to(DEVICE)
            
            logits, query_emb, support_emb, relation_scores = model(
                support_images, support_labels_mapped, query_images
            )
            
            rel_loss = relation_criterion(relation_scores, query_labels_mapped, support_labels_mapped)
            ce_loss = F.cross_entropy(logits, query_labels_mapped)
            total_loss = rel_loss + ce_loss
            
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            probs = F.softmax(logits, dim=1).cpu().numpy()
            
            all_loss.append(total_loss.item())
            all_acc.append((preds == query_labels_mapped.cpu().numpy()).mean())
            
            y_true.extend(query_labels_mapped.cpu().numpy().tolist())
            y_pred.extend(preds.tolist())
            y_prob.extend(probs.tolist())
    
    metrics = compute_all_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob), CLASS_NAMES)
    metrics['test_loss'] = float(np.mean(all_loss))
    metrics['test_acc'] = float(np.mean(all_acc))
    
    return metrics

## 9. Visualization Functions

In [ ]:
# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def plot_training_history(history, save_path=None):
    """Plot training and validation loss/accuracy curves."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    epochs = range(1, len(history['train_loss']) + 1)
    
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
    axes[0, 1].plot(epochs, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title('Training and Validation Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].plot(epochs, history['train_relation_loss'], 'b-', label='Train Rel Loss', linewidth=2)
    axes[1, 0].plot(epochs, history['val_relation_loss'], 'r-', label='Val Rel Loss', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('MSE Loss')
    axes[1, 0].set_title('Relation Loss (MSE)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].bar(epochs, history['epoch_times'], color='green', alpha=0.7)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Time (seconds)')
    axes[1, 1].set_title('Training Time per Epoch')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


def plot_confusion_matrix(cm, class_names, save_path=None, normalize=True):
    """Plot confusion matrix with per-class labels."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    if normalize:
        cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)
        fmt = '.2%'
    else:
        cm_normalized = cm
        fmt = 'd'
    
    sns.heatmap(cm_normalized, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


def plot_per_class_metrics(metrics, class_names, save_path=None):
    """Plot per-class precision, recall, and F1-score."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    x = np.arange(len(class_names))
    width = 0.25
    
    axes[0].bar(x, metrics['precision_per_class'], width, label='Precision', color='steelblue')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Score')
    axes[0].set_title('Per-Class Precision')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([c[:10] for c in class_names], rotation=45, ha='right')
    axes[0].set_ylim([0, 1])
    axes[0].grid(True, alpha=0.3, axis='y')
    
    axes[1].bar(x, metrics['recall_per_class'], width, label='Recall', color='forestgreen')
    axes[1].set_xlabel('Class')
    axes[1].set_ylabel('Score')
    axes[1].set_title('Per-Class Recall')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([c[:10] for c in class_names], rotation=45, ha='right')
    axes[1].set_ylim([0, 1])
    axes[1].grid(True, alpha=0.3, axis='y')
    
    axes[2].bar(x, metrics['f1_per_class'], width, label='F1-Score', color='coral')
    axes[2].set_xlabel('Class')
    axes[2].set_ylabel('Score')
    axes[2].set_title('Per-Class F1-Score')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels([c[:10] for c in class_names], rotation=45, ha='right')
    axes[2].set_ylim([0, 1])
    axes[2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

## 10. XAI Visualization Generation

In [ ]:
# ============================================================
# XAI VISUALIZATION
# ============================================================

def generate_xai_explanations(model, df_test, n_samples=5, save_dir=XAI_DIR):
    """Generate XAI explanations for test samples."""
    ds = ImagePathsDataset(df_test, transform=eval_transform)
    sampler = RelationSampler(df_test['label'].to_numpy(), n_way=N_WAY, k_shot=K_SHOT, q_query=11, episodes=1)
    
    support_idx, query_idx = next(iter(sampler))
    
    support_images = torch.stack([ds[i][0] for i in support_idx]).to(DEVICE)
    support_labels = torch.tensor([ds[i][1] for i in support_idx], dtype=torch.long).to(DEVICE)
    
    unique = torch.unique(support_labels)
    label_map = {int(c): i for i, c in enumerate(unique)}
    support_labels_mapped = torch.tensor([label_map[int(l)] for l in support_labels], dtype=torch.long).to(DEVICE)
    
    query_images = torch.stack([ds[i][0] for i in query_idx]).to(DEVICE)
    query_labels = torch.tensor([ds[i][1] for i in query_idx], dtype=torch.long).to(DEVICE)
    
    print(f"\n{'='*60}")
    print("XAI VISUALIZATION GENERATION (Relation Network)")
    print(f"{'='*60}\n")
    
    sparsity_results = []
    
    for idx in range(min(n_samples, len(query_idx))):
        img = query_images[idx]
        true_label = query_labels[idx].item()
        
        logits, query_emb, support_emb, similarities = model(
            support_images, support_labels_mapped, img.unsqueeze(0)
        )
        pred = torch.argmax(logits, dim=1).item()
        confidence = F.softmax(logits, dim=1)[0, pred].item()
        pred_original = int(list(label_map.keys())[list(label_map.values()).index(pred)])
        
        target_layer = model.feature_encoder.conv_blocks[4]
        
        gradcam = RelationGradCAM(model, target_layer=target_layer,
                                support_images=support_images, support_labels=support_labels_mapped)
        cam_mask = gradcam.generate(img, target_class=pred)
        gradcam.close()
        
        sal_map = relation_saliency_map(model, img, support_images=support_images,
                                       support_labels=support_labels_mapped, target_class=pred)
        
        sparsity_cam = compute_attribution_sparsity(cam_mask)
        sparsity_sal = compute_attribution_sparsity(sal_map)
        
        sparsity_results.append({
            'sample': idx,
            'true_label': CLASS_NAMES[true_label] if true_label < len(CLASS_NAMES) else f'Class {true_label}',
            'pred_label': CLASS_NAMES[pred_original] if pred_original < len(CLASS_NAMES) else f'Class {pred_original}',
            'confidence': confidence,
            'sparsity_cam': sparsity_cam,
            'sparsity_sal': sparsity_sal
        })
        
        save_heatmap(img.cpu(), cam_mask, 
                     os.path.join(save_dir, f'relation_gradcam_sample{idx}_true{true_label}_pred{pred_original}.png'),
                     title=f'Relation Grad-CAM: True={true_label}, Pred={pred_original}')
        
        save_heatmap(img.cpu(), sal_map, 
                     os.path.join(save_dir, f'relation_saliency_sample{idx}_true{true_label}_pred{pred_original}.png'),
                     title=f'Relation Saliency: True={true_label}, Pred={pred_original}')
        
        print(f"Sample {idx}: True={true_label}, Pred={pred_original}, "
              f"Conf={confidence:.3f}, CAM sparsity={sparsity_cam:.3f}, Saliency sparsity={sparsity_sal:.3f}")
    
    return sparsity_results

## 11. Main Execution Pipeline

In [ ]:
# ============================================================
# MAIN EXECUTION
# ============================================================

def main():
    """Execute the complete Relation Network few-shot learning pipeline."""
    print(f"\n{'='*70}")
    print(f"RELATION NETWORKS FOR FEW-SHOT LEARNING WITH XAI")
    print(f"{'='*70}\n")
    
    start_time = time.time()
    
    # STEP 1: DATA SPLITTING
    print("\n" + "="*50)
    print("STEP 1: DATA PREPARATION")
    print("="*50)
    df_train, df_val, df_test, CLASS_NAMES = make_stratified_splits(DATA_ROOT)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    class_counts = df_train['class'].value_counts().sort_index()
    ax.bar(range(len(CLASS_NAMES)), class_counts.values, color='steelblue', alpha=0.8)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels([c[:15] for c in CLASS_NAMES], rotation=45, ha='right')
    ax.set_ylabel('Number of Images')
    ax.set_title('Training Set Class Distribution')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'class_distribution.png'), dpi=150)
    plt.show()
    plt.close()
    
    # STEP 2: MODEL TRAINING
    print("\n" + "="*50)
    print("STEP 2: MODEL TRAINING (Relation Network)")
    print("="*50)
    
    model = RelationNetwork().to(DEVICE)
    print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters\n")
    
    history, best_ckpt = train_relation_network(model, df_train, df_val, n_epochs=NUM_EPOCHS, lr=LEARNING_RATE)
    
    plot_training_history(history, save_path=os.path.join(PLOTS_DIR, 'training_history.png'))
    
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'relation_final.pth'))
    with open(os.path.join(OUTPUT_DIR, 'train_history.json'), 'w') as f:
        json.dump(history, f, indent=2)
    
    model.load_state_dict(torch.load(best_ckpt)['model_state'])
    
    # STEP 3: MODEL EVALUATION
    print("\n" + "="*50)
    print("STEP 3: MODEL EVALUATION")
    print("="*50)
    
    test_metrics = evaluate_relation_network(model, df_test)
    
    print_metrics(test_metrics, CLASS_NAMES)
    
    with open(os.path.join(OUTPUT_DIR, 'test_metrics.json'), 'w') as f:
        json.dump(test_metrics, f, indent=2)
    
    plot_confusion_matrix(np.array(test_metrics['confusion_matrix']), CLASS_NAMES,
                          save_path=os.path.join(PLOTS_DIR, 'confusion_matrix.png'), normalize=True)
    
    plot_per_class_metrics(test_metrics, CLASS_NAMES,
                           save_path=os.path.join(PLOTS_DIR, 'per_class_metrics.png'))
    
    # STEP 4: XAI VISUALIZATION
    print("\n" + "="*50)
    print("STEP 4: XAI VISUALIZATION")
    print("="*50)
    
    sparsity_results = generate_xai_explanations(model, df_test, n_samples=6)
    
    with open(os.path.join(OUTPUT_DIR, 'xai_results.json'), 'w') as f:
        json.dump(sparsity_results, f, indent=2)
    
    total_time = time.time() - start_time
    
    print(f"\n{'='*70}")
    print(f"PIPELINE COMPLETE")
    print(f"{'='*70}")
    print(f"Total execution time: {total_time/60:.1f} minutes")
    print(f"\nFinal Results:")
    print(f"  Test Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"  Test F1-Macro: {test_metrics['f1_macro']:.4f}")
    print(f"  ECE: {test_metrics['ece']:.4f}")
    print(f"\nOutputs saved to: {OUTPUT_DIR}")
    print(f"{'='*70}\n")
    
    return model, test_metrics, history

# Run main pipeline
model, test_metrics, history = main()

## 12. Final Results Summary and Conclusions

In [ ]:
# ============================================================
# FINAL RESULTS
# ============================================================

with open(os.path.join(OUTPUT_DIR, 'test_metrics.json'), 'r') as f:
    final_metrics = json.load(f)

print(f"\n{'='*70}")
print(f"FINAL EVALUATION RESULTS")
print(f"{'='*70}")
print(f"\n{'Metric':<30} {'Value':>15}")
print(f"{'-'*45}")
print(f"{'Test Accuracy':<30} {final_metrics['accuracy']:>15.4f}")
print(f"{'Test F1-Score (Macro)':<30} {final_metrics['f1_macro']:>15.4f}")
print(f"{'Test F1-Score (Micro)':<30} {final_metrics['f1_micro']:>15.4f}")
print(f"{'Test F1-Score (Weighted)':<30} {final_metrics['f1_weighted']:>15.4f}")
print(f"{'Expected Calibration Error':<30} {final_metrics['ece']:>15.4f}")
print(f"\n{'='*70}")
print(f"\nPer-Class Performance:")
print(f"\n{'Class':<25} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"{'-'*55}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name[:25]:<25} "
          f"{final_metrics['precision_per_class'][i]:>10.4f} "
          f"{final_metrics['recall_per_class'][i]:>10.4f} "
          f"{final_metrics['f1_per_class'][i]:>10.4f}")
print(f"{'='*70}\n")

## 13. Research Contributions and Future Work

### 13.1 Summary of Contributions

This implementation demonstrates a complete Relation Network for few-shot learning with:

1. **Learned Comparison Function**: Unlike Siamese networks with fixed metrics, Relation Networks learn to compare images

2. **Relation Module Architecture**: CNN-based module that processes concatenated features

3. **Dual Loss Training**: MSE for relation learning + Cross-entropy for classification

4. **XAI Integration**: Grad-CAM and saliency maps for interpretable predictions

5. **Comprehensive Evaluation**: Accuracy, F1-score, ECE, and attribution sparsity

### 13.2 Key Mathematical Equations

- **Feature extraction**: $\mathbf{F} = f_\phi(\mathbf{x})$
- **Feature concatenation**: $\mathbf{C} = [\mathbf{F}_q; \mathbf{F}_s]$
- **Relation score**: $r = g_\theta(\mathbf{C}) \in [0, 1]$
- **MSE loss**: $L_{relation} = \frac{1}{N} \sum (r_{pred} - r_{target})^2$

### 13.3 Comparison with Other Approaches

| Approach | Comparison Method | Learnable? |
|----------|------------------|------------|
| Prototypical Networks | Euclidean distance to prototypes | No |
| Siamese Networks | Euclidean/cosine similarity | No |
| **Relation Networks** | **Learned neural network** | **Yes** |

### 13.4 References

1. Sung, F., et al. (2018). Learning to Compare: Relation Network for Few-Shot Learning. CVPR.
2. Selvaraju, R. R., et al. (2017). Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization. ICCV.

In [ ]:
# ============================================================
# SAVE ALL RESULTS
# ============================================================

summary = {
    'experiment_config': {
        'n_way': N_WAY,
        'k_shot': K_SHOT,
        'q_query': Q_QUERY,
        'feature_dim': FEATURE_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'train_size': len(df_train),
        'val_size': len(df_val),
        'test_size': len(df_test),
    },
    'final_metrics': final_metrics,
    'class_names': CLASS_NAMES,
    'output_directory': OUTPUT_DIR
}

with open(os.path.join(OUTPUT_DIR, 'experiment_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*70)
print("GENERATED FILES")
print("="*70)

for dir_path, dir_name in [(SPLIT_DIR, 'Splits'), (PLOTS_DIR, 'Plots'), 
                            (XAI_DIR, 'XAI'), (CKPT_DIR, 'Checkpoints')]:
    print(f"\n{dir_name} Directory ({dir_path}):")
    if os.path.exists(dir_path):
        for f in sorted(os.listdir(dir_path)):
            print(f"  - {f}")

print(f"\n{'='*70}")
print("NOTEBOOK EXECUTION COMPLETE")
print(f"{'='*70}\n")